# Plot of the 1-D marginal posteriors for all rounds.

In this notebook, we reproduce Figures 5 and 12 from Ronchi et al. (2026) showing the marginal posterior distribution estimated in all training rounds of the TSNPE algorithm. In particular we consider these two experiments:

- When the entire observed X-ray population is considered (for Figure 12) we use the training results saved on the PIC server at the following path: `/data/magnesia/common/paper_ronchi_etal_2025/B_double_lognorm_dip-tor_heavy/tsnpe_experiment_1_maps8_res32/learning/models/SBI_ConvolutionMDN/20260115_142235/`.
- When only the sample of young magnetars and XDINSs is considered for inference (for Figure 5) we use the training results saved on the PIC server at the following path: `/data/magnesia/common/paper_ronchi_etal_2025/B_double_lognorm_dip-tor_heavy/tsnpe_experiment_1_maps8_res32_youngxdins/learning/models/SBI_ConvolutionMDN/20260202_122351/`.

Note that in order to make this notebook work, you first need to download the results data from `/data/magnesia/common/paper_ronchi_etal_2025/experiments_paper.zip` unpack the file and copy the entire folder experiments into `MAGNESIA_population_synthesis/data/paper_results/ronchi_etal_2026/experiments`. 

In [ ]:
import torch
import corner
import matplotlib.pyplot as plt
import os
import numpy as np
import pandas as pd
import json

from matplotlib import rcParams
from matplotlib import rc
import matplotlib as mpl

#rc("text", usetex=True)
rc("font", family="serif")
#mpl.rcParams["text.latex.preamble"] = r"\usepackage{amsmath}"

In [ ]:
SMALL_SIZE = 30
MEDIUM_SIZE = 50
BIGGER_SIZE = 60

plt.rc("font", size=SMALL_SIZE)  
plt.rc("axes", titlesize=MEDIUM_SIZE)  
plt.rc("axes", labelsize=MEDIUM_SIZE)  
plt.rc("xtick", labelsize=SMALL_SIZE)  
plt.rc("ytick", labelsize=SMALL_SIZE)  
plt.rc("legend", fontsize=SMALL_SIZE)  
plt.rc("figure", titlesize=MEDIUM_SIZE)  

In [ ]:
def import_statistics(stats_path: str):
    """
    Extracting the mean and standard deviation for all the parameters in the `stats_path` file.
    Args:
        stats_path (str): Path to the file where the statistics are saved.
    Returns:
        (torch.tensor, torch.tensor): Mean and standard deviation for the parameters in the `stats_path` file.
    """
    std_list = []
    mean_list = []
    max_list = []
    min_list = []
    with open(stats_path, "r") as json_file:
        data = json.load(json_file)
    for key, value in data.items():
        std_list.append(value["std"])
        mean_list.append(value["mean"])
        max_list.append(value["max"])
        min_list.append(value["min"])
    mean = np.array(mean_list)
    std = np.array(std_list)
    max_list = np.array(max_list)
    min_list = np.array(min_list)
    return mean, std, max_list, min_list

Chose the experiment and rounds to be plotted.

In [ ]:
n_rounds = 10

In [ ]:
# By default we consider the posterior inferred using entire X-ray sample, to reproduce Figure 12.
# To consider the inference results using only young magnetars and XDINSs and reproduce Figure 6 set `use_young_xdins_only` to True.
use_young_xdins_only = True

if use_young_xdins_only:
    stats_path = "../../data/paper_results/paper_ronchi_etal_2026/experiments/tsnpe_experiment_1_maps8_res32_youngxdins/data/statistics_train.json"
    directory_path = "../../data/paper_results/paper_ronchi_etal_2026/experiments/tsnpe_experiment_1_maps8_res32_youngxdins/learning/models/SBI_ConvolutionMDN/20260202_122351"
else:
    stats_path = "../../data/paper_results/paper_ronchi_etal_2026/experiments/tsnpe_experiment_1_maps8_res32/data/statistics_train.json"
    directory_path = "../../data/paper_results/paper_ronchi_etal_2026/experiments/tsnpe_experiment_1_maps8_res32/learning/models/SBI_ConvolutionMDN/20260115_142235"

In [ ]:
observed_posterior_round = []
coverage_round = []

for i in range(n_rounds):
    observed_posterior_round.append(torch.load(f"{directory_path}/round_{i}/samples_posterior.pt").detach().cpu().numpy())

In [ ]:
mean, std, par_max, par_min = import_statistics(stats_path)

In [ ]:
limits = [[-1.5, 0.5], [0.1, 1], [12, 13.5], [0.1, 1], [13.5, 15.0], [0.1, 1], [0.1, 1], [-2.0, 0.0], [24.0, 28.0], [0.1, 1]]
x_ticks = [[-1.5, -1, -0.5, 0.0, 0.5], [0.1, 0.5, 0.9], [12.0, 13.0, 13.5], [0.1, 0.5, 0.9], [13.5, 14.0, 15.0], [0.1, 0.5, 0.9], [0.1, 0.5, 0.9], [-2,-1, 0],[24.0,26.0,28.0],[0.1,0.5,0.9]]
parameter_labels = [
    r"$\mu_{\log P}$",
    r"$\sigma_{\log P}$",
    r"$\mu_{\log B, 1}$",
    r"$\sigma_{\log B, 1}$",
    r"$\mu_{\log B, 2}$",
    r"$\sigma_{\log B, 2}$",
    r"$w_{\log B}$",
    r"$a_{\rm late}$",
    r"$\mu_{\log L_0}$",
    r"$\alpha_L$"
]

n_param = np.shape(observed_posterior_round)[2]

In [ ]:
credibility_level = np.linspace(0, 1, 12)

In [ ]:
n_param = np.shape(observed_posterior_round[0])[1]

fig, axs = plt.subplots(n_rounds, n_param, figsize=(27, 15),gridspec_kw={'hspace': 0, 'wspace': 0.2})
for j in range(n_param):
    for i in range(n_rounds):
        observed_posterior = observed_posterior_round[i]
        observed_posterior = observed_posterior * std[0:n_param] + mean[0:n_param]

        # Set the color for each round posterior.
        curve_color = 'tab:blue'  
        
        # Select the correct axis for each round and parameter.
        ax = axs[i][j]
        
        # Calculate the 95% confidence interval using percentiles to plotted is a grey shaded area.
        lower_bound = np.percentile(observed_posterior.T[j], 2.5)
        upper_bound = np.percentile(observed_posterior.T[j], 97.5)

        # Add grey shading for the 95% confidence interval
        ax.axvspan(lower_bound, upper_bound, color='tab:gray', alpha=0.3)
    
        for past_round in range(0,i):
            observed_posterior_past_round = observed_posterior_round[past_round]
            observed_posterior_past_round = observed_posterior_past_round * std[0:n_param] + mean[0:n_param]

            ax.hist(
                observed_posterior_past_round.T[j],
                bins=32,
                color='grey',
                edgecolor='grey',
                alpha = 0.5,
                linewidth=1.5,
                histtype="step",
                density=True,
            )

        ax.hist(
            observed_posterior.T[j],
            bins=32,
            color=curve_color,
            edgecolor=curve_color,
            linewidth=2,
            histtype="step",
            density=True,
        )
        
        # Only add x-axis tick marks (without labels) for all rows except the last.
        if i < n_rounds - 1 and i>0:
            ax.tick_params(axis='x', which='major', length=10, width=1.5,top=True, labeltop=False, bottom=True, labelbottom=False, direction = 'inout')  # For major ticks
            ax.tick_params(axis='x', which='minor', length=6, width=1,top=True, labeltop=False, bottom=True, labelbottom=False, direction = 'inout')  
            ax.set_xticks(x_ticks[j])
            ax.minorticks_on()
        elif i == 0:
            ax.tick_params(axis='x', which='major', length=10, width=1.5,top=False, labeltop=False, bottom=True, labelbottom=False, direction = 'inout')  # For major ticks
            ax.tick_params(axis='x', which='minor', length=6, width=1,top=False, labeltop=False, bottom=True, labelbottom=False, direction = 'inout',grid_color='r', grid_alpha=0.5)  
            ax.set_xticks(x_ticks[j])
            ax.minorticks_on()
        else:
            ax.tick_params(axis='x', which='major', length=10, width=1.5,top=True, labeltop=False, bottom=True, labelbottom=True, direction = 'inout')  # For major ticks
            ax.tick_params(axis='x', which='minor', length=6, width=1,top=True, labeltop=False, bottom=True, labelbottom=True, direction = 'inout')  
            ax.set_xlabel(parameter_labels[j], fontsize=SMALL_SIZE)
            ax.set_xticks(x_ticks[j], x_ticks[j], fontsize=SMALL_SIZE/2)
            ax.minorticks_on()
            
            
        # Remove y-axis ticks for all subplots.
        ax.set_yticks([])
        
        # Only set titles in the top row.
        if i == 0:
            ax.set_title(parameter_labels[j], fontsize=SMALL_SIZE)

        # Add the round number next to each row, rotated parallel to the y-axis.
        fig.text(0.12, 0.83 - (i / 1.28 - 0.18) / n_rounds, f'Round {i+1}', va='center', ha='center',
                 fontsize=18, rotation='vertical')

        # Set limits for the x-axis
        ax.set_xlim(limits[j])
        
        # Automagically incresing the ylimit axis to a 30% more than the data described.
        ax.margins(y=0.3)

if use_young_xdins_only:
    plt.savefig(f'plots/marginal_posterior_allrounds_youngxdins.png',bbox_inches="tight")
else:
    plt.savefig(f'plots/marginal_posterior_allrounds_full.png',bbox_inches="tight")

plt.show()